# Step 1: Data Description and Understanding

Load the source transactions, inspect their structure, and identify data quality issues before cleaning.

**Input:** `data/raw/data.csv` relative to the project root.


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Support execution from either the repository root or the notebooks/ directory.
_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (candidate for candidate in _candidates if (candidate / "data/raw/data.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find data/raw/data.csv. Run this notebook from the repository, "
        "or place the source CSV at <project-root>/data/raw/data.csv."
    )

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "data.csv"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Load the source dataset from the resolved project path.
df = pd.read_csv(RAW_DATA_PATH, encoding="ISO-8859-1")


### About the Dataset

This dataset is a transnational transaction dataset containing all transactions occurring between 01/12/2010 and 09/12/2011 for a UK-based and registered non-store online retail.

* **Source:** The dataset originates from the **Kaggle** platform (commonly referred to as the "Online Retail Dataset").
* **Content:** It contains transaction-level customer purchasing history, including variables such as invoice numbers, stock item codes, item descriptions, quantities, unit prices, customer IDs, and country coordinates.

**Column Descriptions**

Here is a detailed description of each column in the dataset based on our exploration:

* **`InvoiceNo`**: A 6-digit nominal number uniquely assigned to each transaction. If this code starts with letter 'c', it indicates a cancellation.
* **`StockCode`**: A 5-digit nominal number uniquely assigned to each distinct product.
* **`Description`**: Product name (nominal).
* **`Quantity`**: The quantities of each product per transaction (numeric).
* **`InvoiceDate`**: The day and time when each transaction was generated (numeric/datetime).
* **`UnitPrice`**: Product price per unit in sterling (numeric).
* **`CustomerID`**: A 5-digit nominal number uniquely assigned to each customer.
* **`Country`**: The name of the country where each customer resides (nominal).

**Dataset Structure**

Let's examine the data types, non-null values, and overall structure of the dataset.

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
df.describe()


In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.isna().sum()


In [ ]:
df.duplicated().sum()

In [ ]:
df.nunique()

In [ ]:
df['InvoiceNo'].value_counts()



In [ ]:
df['StockCode'].value_counts()


In [ ]:
df['Description'].value_counts()


In [ ]:
df['Quantity'].value_counts()


In [ ]:
df['InvoiceDate'].value_counts()


In [ ]:
df['UnitPrice'].value_counts()


In [ ]:
df['CustomerID'].value_counts()


In [ ]:
df['Country'].value_counts()

In [ ]:
(df['UnitPrice'] < 0).value_counts()

In [ ]:
#show outliers
numeric_cols = ['Quantity', 'UnitPrice']

for col in numeric_cols:
  sns.boxplot(data=df, x=col)
  plt.title(f'{col} Boxplot')
  plt.show()

# Step 2: Identified Data Problems & Anomalies

Based on our initial data exploration, we have identified several issues that need to be addressed during the cleaning phase:

1. **Negative Values & Cancellations (Anomalies)**:
    * **`Quantity`**: Has negative values (minimum is `-80995`), which represent canceled orders, returns, or adjustments.
    * **`UnitPrice`**: Has negative values (minimum is `-11062.06`), which represent adjustments or system corrections. Transactions with a UnitPrice less than 0 are invalid financial records and must be dropped.
    * **Cancellations**: Canceled transactions are explicitly marked by an `InvoiceNo` starting with the letter **'C'**. We need a specific logic to systematically handle and separate these transactions so they don't distort gross sales analysis.
2. **Missing Values**:
    * **`CustomerID`**: Missing 135,080 entries. These represent guest checkouts or unregistered buyers.
    * **`Description`**: Missing 1,454 entries.
3. **Duplicate Rows**: There are 5,268 duplicate transaction records.
4. **Data Type Alignment**: `InvoiceDate` is stored as an `object` (string) instead of a proper datetime type.
5. **Non-Product Codes**: System administrative, fee, and postage transaction codes (such as `POST`, `DOT`, `BANK CHARGES`, `AMAZONFEE`, `CRUK`) exist in the `StockCode` fields and should be filtered out to isolate physical sales transactions.

# Step 3: Data Cleaning

In this step, we will address the anomalies, missing values, and structure issues discovered during our exploratory phase. Cleaning the data ensures our subsequent analyses and modeling are accurate and robust.

### Cleaning Objectives:
1. **Remove Duplicates**: Eliminate the 5,268 duplicate rows to prevent double-counting transactions.
2. **Handle Missing Values**: Decide on a strategy for missing `CustomerID` (e.g., keeping them with a placeholder `99999` to preserve 25% of our transactional sales record) and missing `Description` records.
3. **Handle Cancellations, Returns & Invalid Prices**:
    * *Cancellations Strategy*: Do not delete cancellation records. Instead, identify transactions where `InvoiceNo` starts with **'C'** and explicitly label them in a new `TransactionType` column (categorized as `'Return'` vs. `'Standard'`). This retains the entire transactional database integrity while cleanly separating financial returns from gross revenue calculations.
    * *Invalid Prices Strategy*: Transactions with a `UnitPrice` strictly less than 0 represent invalid records and are dropped from the dataset.
4. **Correct Data Types**: Convert the `InvoiceDate` column from a string (`object`) type to a standard `datetime` format.
5. **Remove Non-Product Codes**: Filter out system and fee-related transactions (e.g., `POST`, `DOT`, `BANK CHARGES`, `AMAZONFEE`, `CRUK`) to isolate physical item transactions.

In [ ]:
df_raw = df.copy() # Re-initialize df from the raw copy to ensure it's not None

In [ ]:
df = df.drop_duplicates()
df = df.reset_index(drop=True)

In [ ]:
#Fix Data Format
#remove extra space
for col in df.columns:
  if df[col].dtype == 'object':
   df[col] = df[col].str.strip()

### Strategy for Handling Missing Values

Based on the analysis of our missing data, we have two different columns with missing values: `CustomerID` (135,037 missing values) and `Description` (1,454 missing values). Here is the recommended strategy for each:

#### 1. Handling Missing `CustomerID`
* **The Problem:** About 25% of our dataset is missing the `CustomerID`. These likely represent guest checkouts or unregistered customers.
* **The Strategy:** **Keep the records, but assign a placeholder ID (e.g., `99999` or `'Unknown'`).**
    * *Why?* If we delete these rows, we lose a quarter of our sales volume and revenue data, which would heavily bias any overall sales analysis.
    * *Exception:* If we are performing strict Customer Relationship Management (CRM) or cohort analysis (like RFM modeling), we should create a separate filtered dataframe excluding these missing values.

#### 2. Handling Missing `Description`
* **The Problem:** 1,454 records are missing the product `Description`.
* **The Strategy:**
    * First, try to **map the descriptions** using the `StockCode` from other matching rows where the description exists.
    * Second, if no match exists, fill them with **`'Unknown Description'`** or drop them if the count is negligibly small and both description and pricing are missing.

In [ ]:
#Handling Missing CustomerID
df['CustomerID'] = df['CustomerID'].fillna(99999)
df['CustomerID'] = df['CustomerID'].astype(int)

In [ ]:
# 1. Map StockCode to Description where descriptions are not null or empty
stock_desc_map = (
    df[df['Description'].notna() & (df['Description'] != '')]
    .drop_duplicates(subset=['StockCode'])
    .set_index('StockCode')['Description']
    .to_dict()
)

# 2. Fill missing descriptions using the mapped StockCodes
missing_desc_mask = df['Description'].isna() | (df['Description'] == '')
df.loc[missing_desc_mask, 'Description'] = df.loc[missing_desc_mask, 'StockCode'].map(stock_desc_map)

# 3. Fill any remaining unmapped missing descriptions with 'Unknown Description'
df['Description'] = df['Description'].fillna('Unknown Description')


In [ ]:
# Create a transaction type column to cleanly separate standard sales and returns/Cancelled inside the same table
df['TransactionType'] = 'Standard'
df.loc[df["InvoiceNo"].str.startswith("C"), 'TransactionType'] = 'Return'

In [ ]:
# Drop records where UnitPrice is strictly less than 0
df = df[df['UnitPrice'] >= 0]

# Display the breakdown
print(df['TransactionType'].value_counts())

In [ ]:
# Define non-product stock codes or keywords that indicate fees/services
non_product_codes = ['POST', 'DOT', 'M', 'D', 'BANK CHARGES', 'AMAZONFEE', 'CRUK', 'S', 'PADS']

# Filter df to exclude non-product transactions
print(f"Rows before filtering non-products: {df.shape[0]}")
df = df[~df['StockCode'].isin(non_product_codes)]
print(f"Rows after filtering non-products: {df.shape[0]}")

In [ ]:
# 1 Convert InvoiceDate to datetime format
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

# Step 4: Data Transformation

Create signed line revenue and date features for transaction analysis. Return rows retain negative revenue so they reduce net revenue.

### Objectives

1. Create `Revenue = Quantity * UnitPrice`.
2. Extract `YearMonth`, `Hour`, and `DayOfWeek` from `InvoiceDate`.


In [ ]:


# 1. Calculate Revenue (Quantity * UnitPrice)
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# 2. Extract datetime features
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')
df['Hour'] = df['InvoiceDate'].dt.hour
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()

# Display the updated dataframe
display(df.head())
print("Current Data Types:\n", df.dtypes)

# Core Business Analytics Questions

Before finalizing the export, we outline key business questions that this cleaned and prepared dataset is now capable of answering:

### 1. Sales & Revenue Analysis
* **What are the monthly sales trends?** Are there specific months showing seasonal spikes in revenue?
* **How do purchase behaviors vary by day of the week and hour of the day?** When are our peak ordering hours?
* **What is the ratio of standard sales to returned transactions**, and how does that impact net vs. gross revenue?

### 2. Product Performance Analysis
* **Which products (StockCodes) are the top revenue drivers?**
* **Which products are most frequently purchased together** (basket analysis)?
* **Which products have the highest cancellation or return rates?**

### 3. Customer & Country Analysis
* **What is the distribution of customer lifetime value (CLV)** based on their total purchasing history?
* **Which countries generate the highest order volume and revenue?** How does the UK compare to international markets?
* **How do registered customer behaviors differ from guest checkouts (unregistered customers)?**


In [ ]:
# What are the monthly sales trends? Are there specific months showing seasonal spikes in revenue?
print(df['YearMonth'].value_counts())
monthly_revenue = (
    df.groupby(df["InvoiceDate"].dt.to_period("M"))["Revenue"]
      .sum()
)
monthly_revenue.plot(kind="bar", title="Monthly Revenue")
plt.xlabel("Month")
plt.ylabel("Total Revenue")
plt.show()
print('monthly sales:\n', monthly_revenue.sort_values(ascending=False).head(3))


In [ ]:
#How do purchase behaviors vary by day of the week and hour of the day? When are our peak ordering hours?
daily_sales = df.groupby('DayOfWeek')['Quantity'].sum()
hourly_sales = df.groupby('Hour')['Quantity'].sum()
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
daily_sales.plot(kind='bar', title='Daily Sales')
plt.xlabel('Day of the Week')
plt.ylabel('Total Sales')
plt.subplot(1, 2, 2)
hourly_sales.plot(kind='bar', title='Hourly Sales')
plt.xlabel('Hour of the Day')
plt.ylabel('Total Sales')
plt.tight_layout()
plt.show()

print('daily sales: \n', daily_sales.sort_values(ascending=False).head(3))
print('hourly sales: \n', hourly_sales.sort_values(ascending=False).head(3))

In [ ]:
#What is the ratio of standard sales to returned transactions, and how does that impact net vs. gross revenue?

# 1. Calculate overall performance metrics
total_revenue = df[df['TransactionType'] == 'Standard']['Revenue'].sum()
total_quantity = df[df['TransactionType'] == 'Standard']['Quantity'].sum()
total_orders = df[df['TransactionType'] == 'Standard']['InvoiceNo'].nunique()
average_order_value = total_revenue / total_orders if total_orders > 0 else 0

transaction_counts = df['TransactionType'].value_counts()
standard_count = transaction_counts.get('Standard', 0)
return_count = transaction_counts.get('Return', 0)
count_ratio = standard_count / return_count if return_count > 0 else np.nan

# 2. Financial Metrics: Gross vs. Net Revenue
# Gross Revenue comes from Standard transactions
gross_revenue = df[df['TransactionType'] == 'Standard']['Revenue'].sum()
# Returned Revenue comes from Return transactions
returned_revenue = df[df['TransactionType'] == 'Return']['Revenue'].sum()
# Net Revenue is the sum of both (returned_revenue is negative)
net_revenue = gross_revenue + returned_revenue
percent_lost_to_returns = (abs(returned_revenue) / gross_revenue) * 100



# Print the results
print("=== Transaction Counts & Ratio ===")
print(f"Standard Sales Count: {standard_count:,}")
print(f"Returned/Canceled Count: {return_count:,}")

print("\n=== Financial Impact Analysis ===")
print(f"Gross Revenue (Standard Sales):  {gross_revenue:,.2f}")
print(f"Returned/Canceled Amount:        {returned_revenue:,.2f}")
print(f"Net Revenue:                    {net_revenue:,.2f}")
print(f"Percentage of Gross Revenue Lost to Returns: {percent_lost_to_returns:.2f}%\n")
print(f"For every 1 return, we have {count_ratio:.2f} standard transactions.\n")


# Print overall KPIs
sales_summary = pd.DataFrame({
    "Metric": [
        "Total Quantity Sold",
        "Gross Sales Revenue",
        "Return Value",
        "Net Revenue",
        "Total Orders Orders",
        "Average Order Value"
    ],
    "Value": [
        total_quantity,
        gross_revenue,
        returned_revenue,
        net_revenue,
        total_orders,
        average_order_value
    ]
})

print('sales_summary : \n')
print(sales_summary, '\n')


#3. Visualization of Gross vs. Net Revenue
plt.figure(figsize=(8, 6))
revenue_metrics = pd.Series({
    'Gross Revenue': gross_revenue,
    'Net Revenue': net_revenue,
    'Amount Returned': abs(returned_revenue)
})
revenue_metrics.plot(kind='bar', color=['#2ca02c', '#1f77b4', '#d62728'])
plt.title('Financial Impact of Returns on Total Revenue')
plt.ylabel('Amount in ')
plt.xticks(rotation=0)

# Add values on top of the bars
for i, val in enumerate(revenue_metrics):
    plt.text(i, val + (gross_revenue * 0.01), f"{val:,.2f}", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

# 4. Visualize the monthly progression of Orders and Average Order Value
monthly_perf = df[df['TransactionType'] == 'Standard'].groupby('YearMonth').agg(
    Monthly_Revenue=('Revenue', 'sum'),
    Order_Count=('InvoiceNo', 'nunique')
).reset_index()

monthly_perf['AOV'] = monthly_perf['Monthly_Revenue'] / monthly_perf['Order_Count']
monthly_perf['YearMonth_Str'] = monthly_perf['YearMonth'].astype(str)

fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot Number of Orders
color = '#1f77b4'
ax1.set_xlabel('Month')
ax1.set_ylabel('Number of Orders', color=color)
bars = ax1.bar(monthly_perf['YearMonth_Str'], monthly_perf['Order_Count'], color=color, alpha=0.6, label='Number of Orders')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticklabels(monthly_perf['YearMonth_Str'], rotation=45)

# Instantiate a second axes that shares the same x-axis for Average Order Value (AOV)
ax2 = ax1.twinx()
color = '#ff7f0e'
ax2.set_ylabel('Average Order Value (£)', color=color)
line = ax2.plot(monthly_perf['YearMonth_Str'], monthly_perf['AOV'], color=color, marker='o', linewidth=2, label='AOV (£)')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Monthly Unique Orders and Average Order Value (AOV) Over Time')
fig.tight_layout()
plt.show()

In [ ]:
#Which products (StockCodes) are the top revenue drivers?
top_product_sales = df.groupby(['StockCode', 'Description'])['Revenue'].sum().sort_values(ascending=False)
top_product_sales.head(10)

#Which products are most frequently purchased together (basket analysis)?
top_products_sales_togather = df.groupby(['StockCode', 'Description'])['Quantity'].sum().sort_values(ascending=False)
top_products_sales_togather.head(10)

#Which products have the highest cancellation or return rates?
top_products_return = df[df['TransactionType'] == 'Return'].groupby(['StockCode', 'Description'])['Quantity'].sum().sort_values(ascending=False)
top_products_return.head(10)

print('top_product_sales : \n',top_product_sales.head(10), '\n')
print('top_products_sales_togather : \n',top_products_sales_togather.head(10), '\n')
print('top_products_return : \n',top_products_return.head(10), '\n')

In [ ]:
#What is the distribution of customer lifetime value (CLV) based on their total purchasing history?
clv_df = df[df['TransactionType'] == 'Standard'].groupby('CustomerID').agg(
    Total_Spent=('Revenue', 'sum'),
    Order_Count=('InvoiceNo', 'nunique'),
    First_Purchase=('InvoiceDate', 'min'),
    Last_Purchase=('InvoiceDate', 'max')
).reset_index()

# Display summary statistics of CLV
print("=== Customer Lifetime Value (CLV) Summary Statistics ===")
print(clv_df['Total_Spent'].describe())

# Plot the distribution of CLV
plt.figure(figsize=(10, 6))
sns.histplot(clv_df['Total_Spent'], bins=50, kde=True, color='purple')
plt.title('Distribution of Customer Lifetime Value (CLV)')
plt.xlabel('Total Revenue per Customer')
plt.ylabel('Number of Customers')
plt.xlim(0, clv_df['Total_Spent'].quantile(0.95)) # Zoom in on the 95th percentile to avoid heavy skewness
plt.show()

#Which countries generate the highest order volume and revenue? How does the UK compare to international markets?
countries_sales = df[df['TransactionType'] == 'Standard'].groupby('Country')['Revenue'].sum().sort_values(ascending=False)
countries_orders = df[df['TransactionType'] == 'Standard'].groupby('Country')['InvoiceNo'].nunique().sort_values(ascending=False)

print('countries_sales : \n',countries_sales.head(10), '\n')
print('countries_orders : \n',countries_orders.head(10), '\n')

countries_sales.plot(kind='bar', title='Total Revenue by Country')
plt.xlabel('Country')
plt.ylabel('Total Revenue')
plt.xticks(rotation=45)
plt.show()


#How do registered customer behaviors differ from guest checkouts (unregistered customers)?
customer_behavior = df.groupby('CustomerID')['TransactionType'].value_counts().unstack().fillna(0)
guest_id = 99999
registered_df = df[df['CustomerID'] != guest_id]
guest_df = df[df['CustomerID'] == guest_id]

# Compare metrics
comparison = pd.DataFrame({
    'Metric': ['Total Transactions', 'Standard Sales', 'Returns', 'Total Quantity Sold', 'Gross Revenue ', 'Average Order Value '],
    'Registered Customers': [
        len(registered_df),
        len(registered_df[registered_df['TransactionType'] == 'Standard']),
        len(registered_df[registered_df['TransactionType'] == 'Return']),
        registered_df[registered_df['TransactionType'] == 'Standard']['Quantity'].sum(),
        registered_df[registered_df['TransactionType'] == 'Standard']['Revenue'].sum(),
        registered_df[registered_df['TransactionType'] == 'Standard']['Revenue'].sum() / registered_df[registered_df['TransactionType'] == 'Standard']['InvoiceNo'].nunique()
    ],
    'Guest Checkouts': [
        len(guest_df),
        len(guest_df[guest_df['TransactionType'] == 'Standard']),
        len(guest_df[guest_df['TransactionType'] == 'Return']),
        guest_df[guest_df['TransactionType'] == 'Standard']['Quantity'].sum(),
        guest_df[guest_df['TransactionType'] == 'Standard']['Revenue'].sum(),
        guest_df[guest_df['TransactionType'] == 'Standard']['Revenue'].sum() / guest_df[guest_df['TransactionType'] == 'Standard']['InvoiceNo'].nunique() if guest_df[guest_df['TransactionType'] == 'Standard']['InvoiceNo'].nunique() > 0 else 0
    ]
})

print("=== Registered Customers vs. Guest Checkouts Comparison ===\n")
print(comparison.to_string(index=False), '\n')

# Plot comparison of Gross Revenue and Average Order Value
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.barplot(data=comparison[comparison['Metric'].isin(['Gross Revenue (£)', 'Total Quantity Sold'])],
            x='Metric', y='Registered Customers', ax=axes[0], color='royalblue')
axes[0].set_title('Registered Customers')
axes[0].tick_params(axis='x', rotation=15)

sns.barplot(data=comparison[comparison['Metric'].isin(['Gross Revenue (£)', 'Total Quantity Sold'])],
            x='Metric', y='Guest Checkouts', ax=axes[1], color='darkorange')
axes[1].set_title('Guest Checkouts (Unregistered)')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

# Step 6: Exporting Cleaned Data

Export the cleaned transaction-level dataset for analysis and dashboarding.


In [ ]:
cleaned_output_path = PROCESSED_DATA_DIR / "online_retail_clean.csv"
df.to_csv(cleaned_output_path, index=False)
print(f"Saved cleaned dataset to: {cleaned_output_path}")


# Step 7: Machine-Learning Feature Preparation

The ML export adds numeric and encoded representations while retaining the original transaction fields. It is intended as an input dataset for downstream tasks such as customer segmentation and customer lifetime value (CLV) modeling; this notebook prepares features but does not train a model.

### Numeric outlier handling

For `Quantity` and `UnitPrice`, calculate the 1st and 99th percentiles across the cleaned transaction rows. Clip values below the 1st percentile to the lower bound and values above the 99th percentile to the upper bound. This two-sided Winsorization limits extreme values while preserving row counts. Original values remain in the unsuffixed columns; clipped values are stored in `Quantity_Capped` and `UnitPrice_Capped`. `Revenue_Capped` is the signed product of those capped fields, so returns remain negative.

### Encoded features

- `TransactionType_Binary`: standard sales = 1; returns = 0.
- `Country_*`: one-hot indicators for country, with one reference category dropped to avoid redundant columns.


In [ ]:
# Calculate the 1st and 99th percentile bounds using all cleaned transactions.
q_low, q_high = df["Quantity"].quantile([0.01, 0.99])
p_low, p_high = df["UnitPrice"].quantile([0.01, 0.99])

# Clip both tails; preserve original fields and keep return revenue signed.
df["Quantity_Capped"] = df["Quantity"].clip(lower=q_low, upper=q_high)
df["UnitPrice_Capped"] = df["UnitPrice"].clip(lower=p_low, upper=p_high)
df["Revenue_Capped"] = df["Quantity_Capped"] * df["UnitPrice_Capped"]

# Encode transaction type and country for downstream modeling.
df["TransactionType_Binary"] = (
    df["TransactionType"].map({"Standard": 1, "Return": 0}).fillna(1).astype(int)
)
df_ml = pd.get_dummies(df, columns=["Country"], drop_first=True, dtype=int)

print("Quantity clip bounds (1st, 99th percentiles):", q_low, q_high)
print("UnitPrice clip bounds (1st, 99th percentiles):", p_low, p_high)
print("ML feature-set shape:", df_ml.shape)
display(
    df_ml[["Quantity_Capped", "UnitPrice_Capped", "Revenue_Capped", "TransactionType_Binary"]]
    .describe()
)


# Step 8: Exporting the Machine-Learning Feature Set

Save the feature-prepared transaction dataset for downstream segmentation, CLV modeling, or other supervised and unsupervised analyses.


In [ ]:
ml_output_path = PROCESSED_DATA_DIR / "online_retail_clean_ml.csv"
df_ml.to_csv(ml_output_path, index=False)
print(f"Saved ML-ready feature set to: {ml_output_path}")
print(f"Final feature-set shape: {df_ml.shape}")
